### **Ingest circuits files**
1. Read file using spark dataframe reader api
2. Add metadata columns
    - source file
    - injestion timestamp
3. write to bronze delta table

In [0]:
%run ../0-common/env-config

In [0]:
%run ../0-common/bronze_helpers

In [0]:
source_file = f"{landing_folfer_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

1. Read csv file

In [0]:
# creating a schema where defining the data types for the columns present in the csv

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField("circuitId", StringType()),
    StructField("url", StringType()),
    StructField("circuitName", StringType()),
    StructField("lat", DoubleType()),
    StructField("long", DoubleType()),
    StructField("locality", StringType()),
    StructField("country", StringType())
])

In [0]:
circuits_df = (
    spark.read.format('csv')
    .option('header', True)
    .option('mode', 'FAILFAST')
    .schema(circuits_schema)
    .load(source_file)
    )

2. Add metadata columns

In [0]:
circuits_df_final = add_ingestion_metadata(circuits_df)

display(circuits_df_final)

3. Write to bronze delta table

In [0]:
(
    circuits_df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
describe history formula1.bronze.circuits

In [0]:
%sql
select * from formula1.bronze.circuits
version as of 1;